In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import pickle
import time
from tqdm.notebook import tqdm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
# rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

In [ ]:
from dmpe.data_management import DataPaths

from dmpe.utils.density_estimation import (
    update_density_estimate_single_observation,
    update_density_estimate_multiple_observations,
    DensityEstimate,
    get_uniform_target_distribution,
)
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence, valid_space_grid

from dmpe.utils.sets.shared import check_in_set, load_results, DiscretizedSet, SlicedSet, load_discretized_set, save_discretized_set
from dmpe.utils.density_estimation import build_grid, select_bandwidth

from dmpe.evaluation.metrics_utils import default_jsd, default_ae, default_mcudsa, default_ksfc, default_df
from dmpe.evaluation.experiment_utils import extract_metrics_over_timesteps, extract_metrics_over_timesteps_via_interpolation, get_experiment_ids

In [ ]:
def extract_results(lengths, raw_results_path, algo_names, interpolate_to_lengths, system_name, metrics=None, extra_folders=None):

    all_results_by_metric = {}
    
    for (algo_name, use_interpolation) in zip(algo_names, interpolate_to_lengths):
        full_results_path = raw_results_path / pathlib.Path(algo_name) / pathlib.Path(system_name)
        full_results_path = full_results_path / pathlib.Path(extra_folders) if extra_folders is not None else full_results_path

        print("Extract results for", algo_name, "\n at", full_results_path)

        if not use_interpolation:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                lengths=lengths,
                metrics=metrics,
            )
        else:
            all_results_by_metric[algo_name] = extract_metrics_over_timesteps_via_interpolation(
                experiment_ids=get_experiment_ids(full_results_path),
                results_path=full_results_path,
                target_lengths=lengths,
                metrics=metrics,
            )
        print("\n")
    return all_results_by_metric

In [ ]:
#lengths = jnp.linspace(1000, 15000, 3, dtype=jnp.int32)
lengths = jnp.linspace(1000, 15000, 15, dtype=jnp.int32)
lengths

In [ ]:
bandwidth = select_bandwidth(2, 2, 50, 0.3).item()
bandwidth

## fluid_tank:

In [ ]:
# S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "fluid_tank_S_xu_666a769d-8c1b-4b.json")

In [ ]:
# # prepare metrics:

# # jsd:
# points_per_dim = 50
# bandwidth = select_bandwidth(2, 2, 50, 0.3).item()

# check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
# target_distribution_p = get_uniform_target_distribution(
#     dim=2,
#     points_per_dim=points_per_dim,
#     bandwidth=bandwidth,
#     grid_extend=1.0,
#     consider_action_distribution=True,
#     penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu(jnp.concatenate([x, u], axis=-1))),
#     obs_dim=1,
#     act_dim=1,
# )
# jsd_performance_metric = partial(default_jsd, points_per_dim=points_per_dim, bandwidth=bandwidth, target_distribution=target_distribution_p, ca=True)

# # mcudsa:
# points_per_dim = 50
# support_points_mcudsa = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=2,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# mcudsa_performance_metric = partial(default_mcudsa, points_per_dim=None, support_points=support_points_mcudsa)

# # ksfc:
# support_points_ksfc = support_points_mcudsa
# ksfc_performance_metric = partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points_ksfc)

# # df
# points_per_dim = 50
# dim = 2

# support_points = build_grid(dim, low=-1, high=1, points_per_dim=points_per_dim)
# support_spacing_df = jnp.abs(support_points[0] - support_points[1])[-1] / 2

# support_points_df = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=2,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# df_performance_metric = partial(default_df, points_per_dim=None, support_points=support_points_df, support_spacing=support_spacing_df)

# metrics = {
#     "jsd": jsd_performance_metric,
#     "mcudsa": mcudsa_performance_metric,
#     "ksfc": ksfc_performance_metric,
#     "df": df_performance_metric,
# }

In [ ]:
# system_name = "fluid_tank"

# all_fluid_tank_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics=metrics,
# )
# with open(DataPaths().se_cs_experiments / "reduced_space_fluid_tank_results.pickle", "wb") as handle:
#     pickle.dump(all_fluid_tank_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

## pendulum:

In [ ]:
15**3

In [ ]:
# S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "pendulum_S_xu_02430b86-ae0d-42.json")

In [ ]:
# # prepare metrics:

# # jsd:
# points_per_dim = 50
# bandwidth = select_bandwidth(2, 3, 50, 0.3)

# check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
# target_distribution_p = get_uniform_target_distribution(
#     dim=3,
#     points_per_dim=points_per_dim,
#     bandwidth=bandwidth,
#     grid_extend=1.0,
#     consider_action_distribution=True,
#     penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu(jnp.concatenate([x, u], axis=-1))),
#     obs_dim=2,
#     act_dim=1,
# )
# jsd_performance_metric = partial(default_jsd, points_per_dim=points_per_dim, bandwidth=bandwidth, target_distribution=target_distribution_p, ca=True)

# # mcudsa:
# points_per_dim = 50
# support_points_mcudsa = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=3,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# mcudsa_performance_metric = partial(default_mcudsa, points_per_dim=None, support_points=support_points_mcudsa)

# # ksfc:
# support_points_ksfc = support_points_mcudsa
# ksfc_performance_metric = partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points_ksfc)

# # df
# points_per_dim = 15
# dim = 3

# support_points = build_grid(dim, low=-1, high=1, points_per_dim=points_per_dim)
# support_spacing_df = jnp.abs(support_points[0] - support_points[1])[-1] / 2

# support_points_df = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=dim,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# df_performance_metric = partial(default_df, points_per_dim=None, support_points=support_points_df, support_spacing=support_spacing_df)

# metrics = {
#     "jsd": jsd_performance_metric,
#     "mcudsa": mcudsa_performance_metric,
#     "ksfc": ksfc_performance_metric,
#     "df": df_performance_metric,
# }

In [ ]:
# system_name = "pendulum"

# all_pendulum_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics=metrics,
# )
# with open(DataPaths().se_cs_experiments / "reduced_space_pendulum_results.pickle", "wb") as handle:
#     pickle.dump(all_pendulum_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

## cart_pole:

In [ ]:
# S_xu = load_discretized_set(DataPaths().reach_ci_experiments / "cart_pole_S_xu_8744b5d5-30e6-4b.json")

In [ ]:
# def get_uniform_target_distribution(
#     dim: int,
#     points_per_dim: int,
#     bandwidth: float,
#     grid_extend: float,
#     consider_action_distribution: bool,
#     penalty_function,
#     obs_dim: int,
#     act_dim: int,
# ) -> jax.Array:
#     """Get a uniform target distribution for the DMPE algorithm based on the grid parameters
#     and a penalty function. Only values that are not penalized by the penalty function
#     are targeted in the target distribution, but all of them are to be covered uniformly.

#     Args:
#         dim (int): Number of dimensions for the grid.
#         points_per_dim (int): The number of grid points per dimension. Always identical for each
#             dimension
#         bandwidth (float): The bandwidth of the kernel density estimate
#         grid_extend (float): The extent of the grid in each dimension
#         consider_action_distribution (bool): A flag indicating whether the action distribution
#             is to be considered
#         penalty_function (Callable): The penalty function that is used to determine the
#             valid grid points

#     Returns:
#         The target distribution as a jax.Array with shape (points_per_dim**dim, 1)

#     """
#     z_g = build_grid(dim, low=-grid_extend, high=grid_extend, points_per_dim=points_per_dim)

#     if consider_action_distribution:
#         constr_func = lambda z_g: penalty_function(z_g[..., None, :obs_dim], z_g[..., None, -act_dim:])
#     else:
#         constr_func = lambda z_g: penalty_function(z_g[..., None, :obs_dim], None)

#     ## chunk this part. How did I implement this in the past? 
#     # valid_grid_point = jax.vmap(constr_func, in_axes=0)(z_g) == 0

#     n_grid_points = z_g.shape[0]
#     chunk_size = 20_000
#     out = []
    
#     for i in tqdm(jnp.arange(0, n_grid_points, chunk_size)):
#         out.append(jax.vmap(constr_func, in_axes=0)(z_g[i : min(i + chunk_size, n_grid_points)]) == 0)
#     valid_grid_point = jnp.concatenate(out)
#     ##
    
#     constrained_data_points = z_g[jnp.where(valid_grid_point == True)]

#     target_distribution = DensityEstimate.from_dataset(
#         constrained_data_points[None],
#         z_min=-grid_extend,
#         z_max=grid_extend,
#         points_per_dim=points_per_dim,
#         bandwidth=bandwidth,
#     )
#     return target_distribution.p[0] / jnp.sum(target_distribution.p[0])

In [ ]:
# # prepare metrics:
# dim = 5
# obs_dim = 4
# act_dim = 1

# # jsd:
# points_per_dim = 20
# bandwidth = select_bandwidth(2, 5, 20, 0.1)

# check_in_S_xu = partial(check_in_set, grid=S_xu.grid, mask=S_xu.mask)
# target_distribution_p = get_uniform_target_distribution(
#     dim=dim,
#     points_per_dim=points_per_dim,
#     bandwidth=bandwidth,
#     grid_extend=1.0,
#     consider_action_distribution=True,
#     penalty_function=lambda x, u: jnp.logical_not(check_in_S_xu(jnp.concatenate([x, u], axis=-1))),
#     obs_dim=obs_dim,
#     act_dim=act_dim,
# )
# jsd_performance_metric = partial(default_jsd, points_per_dim=points_per_dim, bandwidth=bandwidth, target_distribution=target_distribution_p, ca=True)

# # mcudsa:
# points_per_dim = 20
# support_points_mcudsa = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=dim,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# mcudsa_performance_metric = partial(default_mcudsa, points_per_dim=None, support_points=support_points_mcudsa)

# # ksfc:
# support_points_ksfc = support_points_mcudsa
# ksfc_performance_metric = partial(default_ksfc, points_per_dim=None, variance=0.1, eps=1e-6, support_points=support_points_ksfc)

# # df
# points_per_dim = 7

# support_points = build_grid(dim, low=-1, high=1, points_per_dim=points_per_dim)
# support_spacing_df = jnp.abs(support_points[0] - support_points[1])[-1] / 2

# support_points_df = valid_space_grid(
#     constraint_function=lambda z: jnp.logical_not(check_in_S_xu(z)),
#     data_dim=dim,
#     points_per_dim=points_per_dim,
#     min_value=-1,
#     max_value=1,
# )
# df_performance_metric = partial(default_df, points_per_dim=None, support_points=support_points_df, support_spacing=support_spacing_df)

# metrics = {
#     "jsd": jsd_performance_metric,
#     "mcudsa": mcudsa_performance_metric,
#     "ksfc": ksfc_performance_metric,
#     "df": df_performance_metric,
# }

In [ ]:
# system_name = "cart_pole"

# all_cart_pole_results_by_metric = extract_results(
#     lengths=lengths,
#     raw_results_path=DataPaths().se_cs_experiments,
#     algo_names=["dmpe", "sgoats", "perfect_model_dmpe", "igoats", "random_walk"],
#     interpolate_to_lengths=[False, False, False, False, False],
#     system_name=system_name,
#     extra_folders=None,
#     metrics=metrics,
# )
# with open(DataPaths().se_cs_experiments / "reduced_space_cart_pole_results.pickle", "wb") as handle:
#     pickle.dump(all_cart_pole_results_by_metric, handle, protocol=pickle.HIGHEST_PROTOCOL)

## Plot all together:

In [ ]:
system_name = "fluid_tank"
with open(DataPaths().se_cs_experiments / "reduced_space_fluid_tank_results.pickle", 'rb') as handle:
    all_fluid_tank_results_by_metric = pickle.load(handle)

system_name = "pendulum"
with open(DataPaths().se_cs_experiments / "reduced_space_pendulum_results.pickle", 'rb') as handle:
    all_pendulum_results_by_metric = pickle.load(handle)

system_name = "cart_pole"
with open(DataPaths().se_cs_experiments / "reduced_space_cart_pole_results.pickle", 'rb') as handle:
    all_cart_pole_results_by_metric = pickle.load(handle)

In [ ]:
all_results = dict(
    fluid_tank=all_fluid_tank_results_by_metric,
    pendulum=all_pendulum_results_by_metric,
    cart_pole=all_cart_pole_results_by_metric,
)

In [ ]:
full_column_width = 18.2
half_column_width = 8.89

import matplotlib.ticker as ticker

def custom_formatter(val, pos):
    if val < 1.0:
        return rf"${val:.2f}$"  
    else:
        return rf"${val:.1f}$"  


def plot_metrics_by_sequence_length_for_all_algos_for_all_systems(
    data_per_algo_per_system,
    lengths,
    algo_names,
    plot_log=True,
):
    systems = list(data_per_algo_per_system.keys())
    metric_keys = list(list(data_per_algo_per_system[systems[0]].values())[0].keys())


    print(systems, metric_keys)
    fig, axs = plt.subplots(len(metric_keys), len(systems), figsize=(full_column_width, 13), sharex=True)

    for sys_idx, system_name in enumerate(systems):
        data_per_algo = data_per_algo_per_system[system_name]
        colors = plt.rcParams["axes.prop_cycle"]()

        for algo_name, data in zip(algo_names, data_per_algo.values()):
            c = next(colors)["color"]
            if c == '#d62728':
                c = next(colors)["color"]
            for metric_idx, metric_key in enumerate(metric_keys):
                
                mean = jnp.nanmean(data[metric_key], axis=0)
                std = jnp.nanstd(data[metric_key], axis=0)


                if algo_name=="$\mathrm{sGOATS}$":
                    style = "dotted"
                elif algo_name=="$\mathrm{iGOATS}$":
                    style = "dashdot"
                elif algo_name=="$\mathrm{DMPE}$":
                    style = "dashed"
                else:
                    style=None
                
                axs[metric_idx, sys_idx].plot(
                    lengths,
                    mean,  # jnp.log(mean) if use_log else mean,
                    label=algo_name if metric_idx == 0 and sys_idx == 0 else None,
                    color=c,
                    linewidth=2.5,
                    linestyle=style,
                )
                axs[metric_idx, sys_idx].fill_between(
                    lengths,
                    mean - std,  # jnp.log(mean - std) if use_log else mean - std,
                    mean + std,  # jnp.log(mean + std) if use_log else mean + std,
                    color=c,
                    alpha=0.1,
                )

    for idx, metric_key in enumerate(metric_keys):
        axs[idx, 0].set_ylabel(f"$\mathcal{{L}}_\mathrm{{{metric_key.upper()}}}$")

    for ax in axs[-1]:
        ax.set_xlabel("$k$")

    for ax_ in axs:
        for ax in ax_:
            ax.grid(True, which="both", alpha=0.3)
            ax.tick_params(which='both', axis="y", direction='in')
            ax.tick_params(which='both', axis="x", direction='in')
        ax_[-1].set_xlim(lengths[0] - 0.02 * lengths[-1], lengths[-1] + 0.02 * lengths[-1])

    legend = fig.legend(
        prop={'size': 8 * 2.54},
        framealpha=0.5,
        loc="center",
        bbox_to_anchor=(0.525, 0.0),
        fancybox=True,
        shadow=False, 
        ncol=len(algo_names)
    )

    for ax, col in zip(axs[0], ["$\mathrm{fluid}$ $\mathrm{tank}$", "$\mathrm{pendulum}$", "$\mathrm{cart}$ $\mathrm{pole}$"]):
        ax.set_title(col)

    if plot_log:
        for ax_ in axs[:-1]:
            for ax in ax_:
                ax.set_yscale('log', base=10)

        for idx_y, ax_ in enumerate(axs):
            for idx_x, ax in enumerate(ax_):
                if idx_y == 0:
                    # jsd
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.05, 0.1, 0.3])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.05, 0.1, 0.5])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.15, 0.25, 0.5])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    pass
                elif idx_y == 1:
                    # MCUDSA
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.02, 0.05, 0.1])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.05, 0.1, 0.5])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.25, 0.35, 0.5])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                elif idx_y == 2:
                    # KSFC
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([12.5, 15, 25])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_ticks([10, 100])
                        ax.set_ylim(9, 200)
                        ax.yaxis.set_minor_locator(ticker.LogLocator(numticks=3))
                    elif idx_x == 2:
                        ax.yaxis.set_ticks([100, 1000])
                        ax.set_ylim(100, 1000)
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                elif idx_y == 3:
                    if idx_x == 0:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.2, 0.5, 0.8])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 1:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.2, 0.5, 0.8])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))
                    elif idx_x == 2:
                        ax.yaxis.set_major_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_minor_formatter(ticker.FuncFormatter(custom_formatter))
                        ax.yaxis.set_ticks([0.35, 0.6, 0.9])
                        ax.yaxis.set_minor_locator(ticker.LinearLocator(numticks=0))

    fig.tight_layout(h_pad=-0.1, w_pad=0.35)
    fig.align_ylabels(axs)
    return fig, legend

In [ ]:
all_cart_pole_results_by_metric.keys()

In [ ]:
targeted_algo_order = ["perfect_model_dmpe", "dmpe", "sgoats", "igoats", "random_walk"]
for sys_name, sys_results in all_results.items():
    all_results[sys_name] = {algo_key: sys_results[algo_key] for algo_key in targeted_algo_order}

In [ ]:
plot_metrics_by_sequence_length_for_all_algos_for_all_systems(
    all_results,
    lengths,
    algo_names=[
        "$\mathrm{PM-DMPE}$",
        "$\mathrm{DMPE}$",
        "$\mathrm{sGOATS}$",
        "$\mathrm{iGOATS}$",
        "$\mathrm{random-walk}$",
    ],
)

#plt.show()
plt.savefig("reduced_space_all_metrics_all_systems_all_algos.pdf", bbox_inches='tight')

In [ ]:
x